# CS50 Lecture 3 Key Notes

## 3.1 Week 3: From Step-by-Step Instructions to Measuring Efficiency

By Week 3, the basic building blocks of a program — variables, conditionals, loops, functions — are taken for granted.

Week 2 took Scratch-derived syntax for granted in C in much the same way.

The focus shifts again. Instead of writing correct code, the emphasis turns to writing *efficient* code.

An **algorithm**, as introduced in Week 0, is step-by-step instructions for solving a problem.

Sorting a collection of data, in the everyday sense, means rearranging it from smallest to largest, alphabetically or by some other ordering rule.

Searching means locating a particular piece of information within a collection.

This lecture's two central problems — search and sort — are described as **canonical**. They are algorithms most computer scientists know, and the kind of question a technical interviewer is likely to ask.

The broader goal extends beyond rote familiarity with specific algorithms. It is to build mental models and methodologies for comparing different approaches to the same problem.

It is also to build a vocabulary — **running time**, **Big O** — for describing how a real-world algorithm scales when translated into code that a computer executes.

## 3.2 The Attendance Problem: Three Algorithms for Counting

### 3.2.1 Counting by Ones

Counting a roomful of people one at a time — 1, 2, 3, 4, ... — takes one step per person.

For `n` people, the algorithm takes `n` steps.

Graphed against the number of people in the room (x-axis) versus the number of steps taken (y-axis), this produces a straight line with slope 1. Here's the part worth holding onto: a straight line on this kind of graph is the visual signature of *linear* growth, and it will reappear throughout this lecture as the marker of an `O(n)` algorithm.

### 3.2.2 Counting by Twos

Counting in increments of two — 2, 4, 6, 8, ... — still takes a number of steps proportional to `n`.

Specifically, it takes `n/2` steps.

It is still graphed as a straight line, just one with a shallower slope.

It is twice as fast as counting by ones for any room size — a room of 100 people takes 100 steps counted by ones, but only 50 steps counted by twos.

Fundamentally, however, it is the same kind of growth: linear. Halving the step count is a constant-factor improvement, not a change in kind, and that distinction matters more than it first appears — it is the same distinction the lecture later draws between an *algorithm* and an *improvement* to one.

### 3.2.3 Divide and Conquer: Pairwise Summation

A third algorithm works differently.

Each of `n` people starts by holding the value 1.

Each then pairs off with another person who is still standing, and the two add their held values together.

One person of each pair sits down holding the sum, while the other remains standing.

Repeating this pairing-and-summing step among those still standing causes the number of people left standing to halve on every repetition.

Eventually a single person remains, holding the running total — the count of the room.

Applied to a room of people, the count obtained this way (**141**) closely matched, but did not exactly equal, a count obtained by counting one person at a time aloud (approximately **160**).

The discrepancy is attributed to small arithmetic errors accumulating during the manual one-at-a-time count, not to a flaw in the pairing algorithm itself. It is worth being explicit about this: the algorithm itself is not in question here, only the unreliability of doing repeated mental addition aloud under time pressure.

This pairing algorithm is far faster than either of the first two for large `n`.

Doubling the number of people requires only one additional pairing round. Quadrupling it requires only two additional rounds.

The relationship between problem size and number of steps is no longer a straight line — it flattens out.

In hindsight, this algorithm is recognizable as the same **divide-and-conquer** idea behind the Week 0 phone-book search. That idea is later formalised in this lecture as **binary search**.

## 3.3 Searching

### 3.3.1 The Search Problem as a Black Box

A search algorithm can be modelled as a black box: it accepts some input and produces an output.

```text
input → [ search algorithm ] → output
```

For the search problem specifically, the input is a collection of values — for example, money sealed behind a row of numbered lockers.

The output is a single boolean: true if a target value is present, false otherwise.

This black-box framing parallels a function definition in R: a defined set of inputs maps to a single returned value, and the caller does not need to know how the function's internals produce that value. The same idea also has a Python equivalent in a function signature, but the R framing is the more natural first reach here, since the input-to-output mapping is exactly how an R function is taught from the very first lesson.

Locations are addressed using zero-indexing. Seven lockers are numbered 0 through 6, not 1 through 7.

Hearing a reference to "location 6" therefore implies at least 7 total locations.

This convention departs from R, where vector indexing starts at 1 rather than 0 — `vec[1]` is R's first element, where the equivalent zero-indexed structure would call it element 0. The convention is, however, the same zero-indexing already familiar from Python list indexing, where `list[0]` is the first element. Worth flagging early: this is one of the rarer cases where R's indexing habit does not transfer, and defaulting to "first element is index 1" out of R habit will be a source of off-by-one errors once C arrays are introduced.

A real computer cannot perceive an entire array at a glance the way a human eye can scan a printed grid of numbers.

Each value behind a closed locker door must be checked one at a time: open the door, read the value, close the door, move to the next.

This constraint is the operating assumption behind all of the algorithms that follow.

### 3.3.2 Linear Search

Consider seven lockers holding unsorted denominations of money.

The simplest strategy for locating a specific bill — for example, USD 50 — is to check each locker in turn, left to right, stopping as soon as it is found.

Demonstrated on lockers holding, in order, USD 20, USD 500, USD 10, USD 5, USD 100, USD 1 and USD 50, the target (USD 50) is not found until the very last locker is opened.

This is the worst case for this approach, since the search must continue until either the value is found or every locker has been checked.

Whether the scan proceeds left-to-right or right-to-left — or in any other fixed, consistent order — makes no fundamental difference if nothing is known about how the values are arranged.

Each ordering finds the target on the first try only by luck.

Jumping around in a random order does not help either. It also requires tracking which lockers have already been checked, which is extra memory overhead that becomes significant as the collection grows from 7 lockers to 700.

#### 3.3.2.1 Linear Search: Success Even Without a Match

A worked example on a plain 15-element array of integers, `11 23 8 14 30 9 6 17 22 28 25 15 7 10 19`, illustrates both outcomes of linear search.

Searching for the target `9`: the scan checks `11`, `23`, `8`, `14`, `30` and finds `9` at the sixth position, stopping immediately.

Searching the same array for `50` (not present): the scan must check all 15 elements.

Every one of `11, 23, 8, 14, 30, 9, 6, 17, 22, 28, 25, 15, 7, 10, 19` is examined before concluding the value is absent.

This second case clarifies what "failure" means for a search algorithm.

Linear search reporting `50` as not found is not a failure of the algorithm.

It did exactly what it was asked: exhaustively check every element, and correctly conclude the target is absent only after ruling out every position.

**Linear search succeeds whether or not the target is present**, so long as it returns the correct boolean answer.

An actual failure would be an *incorrect* answer — for instance, reporting "not found" without having checked every element. That distinction is easy to blur but worth keeping sharp, since the worst-case running time of an algorithm is defined by how much work *correct* behaviour can demand, not by how the algorithm behaves when something goes wrong.

This is the mechanism behind two bounds established later in this lecture: the worst case (`O(n)`, scanning everything to confirm absence or find a value at the very end) and the best case (`Ω(1)`, finding the target on the very first check).

### 3.3.3 Binary Search

Consider the same seven lockers, now sorted from smallest to largest: USD 1, USD 5, USD 10, USD 20, USD 50, USD 100, USD 500.

The search proceeds far more efficiently by checking the middle locker first.

Based on whether the target is smaller or larger than the middle value, the half of the remaining lockers that cannot possibly contain the target is discarded entirely.

The search then repeats on the remaining half.

This **divide-and-conquer** strategy is the same approach used for the phone-book lookup in Week 0 and for the pairwise-counting algorithm earlier in this lecture.

#### 3.3.3.1 Binary Search: Tracking Start, End, and Middle

Implementing binary search concretely requires tracking three values as the search progresses: a **start** index, an **end** index and a **middle** index computed from them.

```text
middle = (start + end) / 2
```

This division is integer division — any fractional remainder is discarded.

Each comparison against the value at `middle` narrows the sub-array by moving one of the two boundaries.

If the target is greater than the middle value, `start` becomes `middle + 1`. If the target is less, `end` becomes `middle - 1`.

Consider the sorted 15-element array `6 7 8 9 10 11 14 15 17 19 22 23 25 28 30` — the same 15 values used in 3.3.2.1's linear-search example, now sorted.

Tracing a search for `19`:

| Step | Start | End | Middle | Value at Middle | Outcome |
|---|---|---|---|---|---|
| 1 | 0 | 14 | 7 | 15 | 19 > 15 → start = 8 |
| 2 | 8 | 14 | 11 | 23 | 19 < 23 → end = 10 |
| 3 | 8 | 10 | 9 | 19 | found |

Three steps locate the target. This is consistent with `O(log n)`, since `log₂15 ≈ 3.9`.

Searching the same array for `16` (not present) illustrates how the algorithm recognises absence:

| Step | Start | End | Middle | Value at Middle | Outcome |
|---|---|---|---|---|---|
| 1 | 0 | 14 | 7 | 15 | 16 > 15 → start = 8 |
| 2 | 8 | 14 | 11 | 23 | 16 < 23 → end = 10 |
| 3 | 8 | 10 | 9 | 19 | 16 < 19 → end = 8 |
| 4 | 8 | 8 | 8 | 17 | 16 < 17 → end = 7 |
| 5 | 8 | 7 | — | — | **start > end** |

> When `start` exceeds `end`, the sub-array being searched has shrunk to size 0 — the two boundaries have "crossed."
>
> This is the operational form of the empty-array guard already stated in 3.4.2's pseudocode ("if there are no doors left, return false").
>
> It is distinct from `start == end`: a single remaining element (step 4 above) is still checked before being ruled out. Only once the boundaries cross can the algorithm conclude the target is absent. Here is the part that is easy to miss on a first read: the crossing is not an edge case bolted onto the algorithm afterward, it is the natural endpoint that the boundary-narrowing logic arrives at on its own once nothing is left to check.

## 3.4 Search Algorithms in Pseudocode

### 3.4.1 Linear Search Pseudocode

The problem to be solved: `Given a row of numbered doors, determine whether the value 50 is behind one of them, and return true or false accordingly.`

In English:

```text
For each door from left to right:
    If 50 is behind the door:
        return true
return false
```

The final `return false` belongs outside and after the loop, at the same indentation level as the `for` — not nested inside an `else` branch attached to the `if`.

A tempting but incorrect variant:

```text
For each door from left to right:
    If 50 is behind the door:
        return true
    Else:
        return false
```

> This version is wrong.
>
> If the very first door checked does not hold the target, the `else` branch returns `false` immediately, without ever examining the remaining doors.
>
> `return` ends a function's execution immediately, so the rest of the loop never runs. The correct version waits until every door has been checked unsuccessfully before returning `false`.

In array notation, with `n` total doors indexed `0` through `n-1`:

```text
For i from 0 to n-1:
    If 50 is behind doors[i]:
        return true
return false
```

The loop bound is `n-1`, not `n`.

With zero-indexing and `n` total elements, the last valid index is `n-1`. A row of 7 doors, for instance, is indexed 0 through 6 — `n-1` where `n` is 7 — and a loop that mistakenly ran through index `n` (here, 7) would be reaching for an eighth door that does not exist.

Pseudocode of this kind functions as a deliberate amalgam of English, C and — later in the course — Python.

Writing the logic out in pseudocode first makes translating it into any specific language more direct, especially as the underlying problem grows more complex. The same habit carries over cleanly to R: sketching the loop and condition in this English-like form first, then filling in R's `for` and `if` syntax afterward, tends to produce fewer bugs than writing R syntax directly from the problem statement.

### 3.4.2 Binary Search Pseudocode

In English:

```text
If 50 is behind the middle door:
    return true
Else if 50 is less than the number behind the middle door:
    search the left half
Else if 50 is greater than the number behind the middle door:
    search the right half
```

A guard for the empty case must be added at the top, since each recursive halving eventually leaves no doors left to search:

```text
If there are no doors:
    return false
If 50 is behind the middle door:
    return true
Else if 50 is less than the number behind the middle door:
    search the left half
Else if 50 is greater than the number behind the middle door:
    search the right half
```

"Search the left half" and "search the right half" each mean applying this same procedure again to a smaller subset of doors.

This self-referential technique is formalised later in the lecture as **recursion**.

In array notation:

```text
If 50 is behind doors[middle]:
    return true
Else if 50 < doors[middle]:
    search doors[0] through doors[middle-1]
Else if 50 > doors[middle]:
    search doors[middle+1] through doors[n-1]
```

The middle index is the number of remaining doors divided by 2, rounded down — integer division.

For 7 doors, `7 / 2 = 3.5`, rounded down to index `3`, the true middle of a 7-element, zero-indexed array.

Both halves deliberately exclude index `middle` itself, since that element was already checked by the preceding `if`. Skipping that excluded middle index is not a minor bookkeeping nicety — omitting it would mean re-checking an element already ruled out, which costs nothing in correctness here but would matter once step counts are being tallied precisely in §3.5.

> **Cheat sheet — linear vs. binary search**

| | Linear search | Binary search |
|---|---|---|
| Precondition | None — works on unsorted data | Data must already be sorted |
| Strategy | Check every element in order | Repeatedly check the middle, discard a half |
| Worst case | `O(n)` | `O(log n)` |
| Best case | `Ω(1)` (target at first position) | `Ω(1)` (target at first middle) |
| Index bookkeeping | Single index, `0` to `n-1` | Three indices: `start`, `end`, `middle` |

## 3.5 Running Time and Big-O Notation

### 3.5.1 What Running Time Measures

**Running time** is not a clock reading in real seconds.

A figure in seconds would depend on the specific machine, its load at the time and incidental implementation details.

None of that describes the algorithm itself.

Instead, an algorithm's running time is expressed as the number of steps it takes: comparisons, swaps or other discrete operations, counted as a function of input size.

This gives computer scientists a shared vocabulary for comparing one algorithm against another, independent of hardware.

Differences between algorithms are invisible on small inputs.

Any algorithm finishes near-instantly on a handful of values — three lockers, five rows, a dozen names.

The differences emerge only when reasoning about what happens as the input size `n` grows much larger: a thousand elements, 100mn elements or 1bn elements.

Running time is therefore analysed in broad strokes, via a graph of step count against problem size, rather than by tallying exact operation counts for one specific `n`.

> An R analogy makes the same point concretely. Checking `target %in% x` (or `match(target, x)`) on a plain, unsorted numeric vector scans element by element, just like linear search — the cost grows with the vector's length, whether `x` holds 10 elements or 10mn. Looking a value up by name instead, in a named vector or a `list()` keyed by name (`lookup[["target"]]`), uses R's internal hashing for that lookup, so the cost stays roughly constant regardless of how many entries the structure holds. Here's the part that is easy to miss: the *algorithm* changes from a scan to a hash lookup, and nothing about CPU speed or R version explains the difference — only the underlying decision procedure does.

> The same contrast holds in Python. Checking `target in my_list` on a plain Python list scans element by element, exactly like linear search. Checking `target in my_set` (or `target in my_dict`) instead uses a hash table, whose lookup cost stays roughly constant no matter how many elements the set holds. It is the same R point in a second language, which is worth seeing twice since the underlying idea — trading a scan for a hash lookup — recurs constantly once searching and sorting are both on the table.

### 3.5.2 Step Counts for Linear Search, Optimized Linear Search, and Binary Search

Three algorithms already introduced supply concrete step counts to compare, in terms of the number of elements `n`:

- Linear search, worst case: `n` steps.
- Linear search, checking only every other locker: `n/2` steps — still proportional to `n`.
- Binary search: `log₂n` steps — the same halving logic behind the pairwise-counting algorithm of 3.2.3, where dividing a problem of size `n` in half repeatedly leaves a single element after `log₂n` divisions.

### 3.5.3 Big O: An Upper Bound on the Dominant Term

**Big O** (`O` = "order of") notation states that an algorithm takes on the order of some number of steps, give or take — not an exact count.

Two simplifications follow from this looseness.

The first simplification drops constant factors.

`O(n)` and `O(n/2)` are both written `O(n)`.

The reasoning runs as follows: as `n` grows large, both expressions grow at the same linear rate, differing only by a fixed multiplier of one-half.

That multiplier does not change the *shape* of the growth, only its steepness.

Concretely, at `n = 1,000,000` the two expressions are `1,000,000` steps and `500,000` steps — very different numbers, but both still scale up by roughly the same factor whenever `n` doubles, which is the property Big O actually cares about.

So Big O keeps the dominant term, `n`, and discards the constant attached to it.

The second simplification drops the base of a logarithm.

`O(log₂n)` is written `O(log n)`.

Changing a logarithm's base is, again, equivalent to multiplying by a constant factor.

Converting between `log₂` and `log₁₀`, for instance, only rescales the curve — at `n = 1,000,000`, `log₂n` is about 20 while `log₁₀n` is exactly 6, a fixed ratio between them at every value of `n`.

Since Big O already discards constant factors, the base is dropped along with them.

> Computer scientists ignore lower-order terms — a divide-by-two, a logarithm's base — and look only at the term that dominates as `n` grows arbitrarily large. That dominant term is what decides how the algorithm actually scales.

#### 3.5.3.1 Visualizing Growth: Straight Lines vs. a Flattening Curve

Plotted with step count on the y-axis and problem size on the x-axis, the `O(n)` and `O(n/2)` curves are straight lines.

The `O(n/2)` line sits at half the slope of `O(n)`.

Both lines remain straight no matter how far the x-axis is zoomed out, from a handful of elements to 1bn.

The `O(log n)` curve, by contrast, flattens further the more it is zoomed out.

Doubling `n` adds only one more step to a binary search — going from 1,000 elements to 2,000 elements costs a single extra comparison, and going from 500mn elements to 1bn elements still costs only one extra comparison.

```text
steps
  |                                          O(n)
  |                                      ,·'
  |                                  ,·'      O(n/2)
  |                              ,·'      ,·'
  |                          ,·'      ,·'
  |                      ,·'      ,·'
  |                  ,·'      ,·'           O(log n)
  |              ,·'      ,·'        ____________
  |          ,·'      ,·'  _______/
  |      ,·' ,·' ____/
  |  ,·'_____/
  +---------------------------------------------------> problem size
```

> A physical analogy for the flattening curve helps make the shape concrete.
>
> Doubling a single line of people waiting at a locker bank doubles the time needed to check each one in turn — twice the people, twice the steps, a straight line.
>
> Doubling the *height* of a single stack of paper, by contrast, does not double the number of times the stack must be cut in half to isolate one sheet. Each additional cut only handles twice as much material as the one before it, so the number of cuts needed grows far more slowly than the stack's height.
>
> Repeated halving is the mechanism behind both binary search and the flattening `O(log n)` curve.

### 3.5.4 A Cheat Sheet of Common Running Times

| Running time | Description |
|---|---|
| `O(1)` | Constant — a fixed number of steps (1, 2, 4, 10, ...) regardless of input size |
| `O(log n)` | Logarithmic |
| `O(n)` | Linear |
| `O(n log n)` | "`n` times `log n`" steps |
| `O(n²)` | Quadratic ("`n` squared") |

### 3.5.5 Big O Applied to Linear and Binary Search

Linear search is **O(n)**.

In the worst case the target sits at the very end of the lockers, or is altogether absent, and every locker must be opened to confirm either outcome.

Getting lucky — finding the target on an early locker — does not change this classification.

Big O is conventionally applied to the **worst case**: the scenario that bounds how badly an algorithm could perform on an unfavourable input.

Binary search is **O(log n)**, provided the data is already sorted.

Halving the remaining lockers on every comparison reaches the target, or exhausts the search space, in `log₂n` steps — the same divide-and-conquer logic as the Week 0 phone book and 3.2.3's pairwise counting.

Binary search depends entirely on the sorted precondition.

Applied to unsorted data, each less-than/greater-than comparison discards a half that may or may not actually contain the target, degrading the search into a sequence of arbitrary, often-wrong eliminations rather than a reliable narrowing.

> R is where this precondition shows up most cleanly, and it is worth dwelling on since it is the same lesson as the lockers, just in vector form.
>
> A sorted numeric vector allows `findInterval()` to locate a value's position by halving, much like binary search narrowing in on a locker.
>
> An unsorted vector forces a linear scan instead, via `which(x == target)`, checking every element in turn — there is no shortcut available, because nothing in the data's arrangement can be trusted to skip half the search space safely.
>
> Sorting the vector first (`sort(x)`) is what unlocks the faster search. That single upfront step is exactly what binary search requires of the lockers before it can discard half the search space on each comparison.

#### 3.5.5.1 When Sorting First Pays Off (and When It Doesn't)

Binary search requires sorted data, and sorting itself costs steps.

Which strategy wins depends on how many times the data will be searched.

Searching a collection exactly once does not justify the cost of sorting it first — plain linear search is cheaper overall in that case.

Searching the same collection many times changes the calculation.

The one-time cost of sorting is paid once and then amortised across every subsequent binary search.

That amortised cost ends up cheaper per search than repeating a linear scan each time — the more searches performed against the same sorted collection, the more that upfront sorting cost pays for itself.

## 3.6 Big Omega, Big Theta, and Asymptotic Notation

### 3.6.1 Big Omega: A Lower Bound

**Big Omega (Ω)** is the lower-bound counterpart to Big O: how few steps an algorithm might take in the best case, if everything goes as luckily as possible.

Linear search is **Ω(1)**.

The reasoning: the target could be behind the very first locker checked, regardless of how many lockers there are in total — one locker out of 10, or one locker out of 10mn, makes no difference to this particular scenario.

A single check is then enough to finish, a fixed number of steps unrelated to `n`.

Binary search is also **Ω(1)**, by the same kind of reasoning.

The target could sit exactly at the middle locker checked on the very first comparison, again finishing in a single step regardless of how large the collection is.

Both algorithms therefore share the same lower bound despite having different upper bounds: `O(n)` for linear search, `O(log n)` for binary search.

Together, the two bounds sketch a range of possible performance — very fast in a lucky best case, but slower toward the worst case.

Here is the part that matters most in practice: an algorithm is not lucky every time it runs, so over many runs its performance tends to sit closer to its upper bound than its lower one.

A single lucky run at `Ω(1)` says very little about what to expect on average.

### 3.6.2 Big Theta: When Upper and Lower Bounds Coincide

**Big Theta (Θ)** is used when an algorithm's upper and lower bounds coincide — when Big O and Big Omega describe the exact same number of steps.

A Θ classification collapses two pieces of information, an upper bound and a lower bound, into one.

Neither linear search nor binary search qualifies for Θ.

Linear search has `O(n)` paired with `Ω(1)`.

Binary search has `O(log n)` paired with `Ω(1)`.

Each has a best case strictly faster than its worst case, so a gap remains between the two bounds.

An algorithm earns a Θ classification only once no such gap remains, that is, only once its best case and worst case require the same number of steps.

### 3.6.3 Asymptotic Notation, Defined

**Asymptotic notation** is the collective term for Big O, Big Omega and Big Theta.

*Asymptotic* describes a quantity that keeps growing as `n` grows arbitrarily large, without necessarily ever reaching a fixed boundary.

The notation describes an algorithm's trajectory at that scale, not an exact step count for any one input size.

It is a description of a trend, not a measurement of a single point.

> **Cheat sheet: Big O / Ω / Θ for linear and binary search**
>
> | Algorithm | Precondition | O (upper bound, worst case) | Ω (lower bound, best case) | Θ (tight bound) |
> |---|---|---|---|---|
> | Linear search | none (works on unsorted data) | `O(n)` | `Ω(1)` | none — `O` and `Ω` don't coincide |
> | Binary search | data must be sorted | `O(log n)` | `Ω(1)` | none — `O` and `Ω` don't coincide |
>
> | Running time | Description |
> |---|---|
> | `O(1)` | Constant — a fixed number of steps regardless of input size |
> | `O(log n)` | Logarithmic |
> | `O(n)` | Linear |
> | `O(n log n)` | "`n` times `log n`" steps |
> | `O(n²)` | Quadratic ("`n` squared") |

## 3.7 Implementing Linear Search in C

### 3.7.1 Searching an Array of Integers

A program, `search0.c`, implements linear search concretely:
```c
#include <cs50.h>
#include <stdio.h>

int main(void)
{
    int numbers[] = {20, 500, 10, 5, 100, 1, 50};
    int n = get_int("Number: ");
    for (int i = 0; i < 7; i++)
    {
        if (numbers[i] == n)
        {
            printf("Found\n");
            return 0;
        }
    }
    printf("Not found\n");
    return 1;
}
```

The array's size (7) is left for the compiler to infer from the number of values inside the curly braces.

Stating the size explicitly in the square brackets is permitted.

Doing so risks a mismatch between the declared size and the actual element count, though, if the two are ever edited independently.

Returning `0` signals success and returning `1` signals failure, following the exit-status convention introduced in Week 2.

Hard-coding the loop bound at `7` is not ideal style. It is acceptable here, though, since the literal is not reused anywhere else in the program.

An array of seven `int`s behaves like a fixed row of seven numbered lockers bolted to a wall. Each locker holds exactly one value, and the row can neither grow a new locker nor shrink one away once built.

`numbers[i]` is the instruction "open locker `i` and read what's inside."

Tracing the program for an input of `50`:

| Step | `i` | `numbers[i]` | Comparison | Action |
|---|---|---|---|---|
| 1 | 0 | 20 | `20 == 50`? No | continue |
| 2 | 1 | 500 | `500 == 50`? No | continue |
| 3 | 2 | 10 | `10 == 50`? No | continue |
| 4 | 3 | 5 | `5 == 50`? No | continue |
| 5 | 4 | 100 | `100 == 50`? No | continue |
| 6 | 5 | 1 | `1 == 50`? No | continue |
| 7 | 6 | 50 | `50 == 50`? Yes | print `Found`, `return 0` |

Seven lockers, seven comparisons at most — and this trace needs all seven, because `50` is sitting in the very last one.

For an input of `20`, the match occurs on the very first iteration (`i = 0`), and the loop exits immediately.

For an input of `1000`, all seven comparisons fail.

The loop's condition `i < 7` becomes false at `i = 7`, control falls through to `printf("Not found\n")`, and the program returns `1`.

### 3.7.2 Searching an Array of Strings: Why `==` Fails

Adapting the same program to search an array of strings — six Monopoly game pieces — surfaces a new bug:
```c
string strings[] = {"battleship", "boot", "cannon", "iron", "thimble", "top hat"};
string s = get_string("String: ");
for (int i = 0; i < 6; i++)
{
    if (strings[i] == s)
    {
        printf("Found\n");
        return 0;
    }
}
```

Running this program reports `"battleship"`, `"boot"`, and `"top hat"` as **not found**, even though each is genuinely present in the array.

Tracing a search for `"boot"` against this array exposes exactly where the comparison goes wrong:

| Step | `i` | `strings[i]` | `strings[i] == s` | What `==` actually compares |
|---|---|---|---|---|
| 1 | 0 | `"battleship"` | false | the memory address of `"battleship"` vs. the memory address of the typed `"boot"` |
| 2 | 1 | `"boot"` | false | the memory address of this `"boot"` vs. the memory address of the typed `"boot"` |

Both strings at step 2 spell the same four letters, yet `==` reports false.

The reason: the two `"boot"` values live at two different locations in memory. One sits inside the hard-coded array; the other sits in the buffer `get_string` allocated for the typed input.

`==` on a `string` compares those two addresses, not the characters stored at them.

> Comparing strings with `==` does not work the way it does for integers. An integer comparison is a single, direct check of two numeric values.
>
> A string is a sequence of characters spread across memory. Determining whether two strings are equal requires checking every character in sequence — a job `==` does not do.

In R, by contrast, `"boot" == "boot"` correctly evaluates to `TRUE`. R's `==` operator performs an element-wise character comparison for strings automatically, the same way it would for an entire character vector, hiding the address-vs-content distinction that C exposes.

Python behaves the same way: `"boot" == "boot"` also evaluates to `True`, for the same underlying reason — the language's `==` is defined to compare string *contents*, never raw memory addresses.

The fix in C is `strcmp`, declared in `string.h` (in the same family as the previously introduced `strlen`).

`strcmp` walks both strings left to right, character by character. It returns `0` if they are equal, a negative number if the first string would come before the second and a positive number if it would come after.

"Before" and "after" here mean **asciibetically** — ordered by the underlying ASCII values of the characters. This only approximates true alphabetical order.
```c
if (strcmp(strings[i], s) == 0)
```

A second bug appears on first attempting this fix: a compiler error reading roughly "call to undeclared library function `strcmp`."

The cause is omitting `#include <string.h>`; adding the header resolves it.

The corrected program:
```c
#include <cs50.h>
#include <stdio.h>
#include <string.h>

int main(void)
{
    string strings[] = {"battleship", "boot", "cannon", "iron", "thimble", "top hat"};
    string s = get_string("String: ");
    for (int i = 0; i < 6; i++)
    {
        if (strcmp(strings[i], s) == 0)
        {
            printf("Found\n");
            return 0;
        }
    }
    printf("Not found\n");
    return 1;
}
```

Tracing this corrected version for an input of `"boot"`: at `i = 0`, `strcmp("battleship", "boot")` returns a nonzero value, since the strings differ.

At `i = 1`, `strcmp("boot", "boot")` walks both strings character by character — `b` vs. `b`, `o` vs. `o`, `o` vs. `o`, `t` vs. `t` — finds no mismatch at any position, and returns `0`.

The `if` condition `strcmp(strings[i], s) == 0` becomes true. `"Found"` is printed, and the program returns `0`.

> Comparing strings with `==` is a common bug. The code compiles and can appear to "almost work," but it silently fails to detect equal strings.
>
> String equality in C always requires `strcmp(a, b) == 0`.

`strcmp`'s three-way return value — negative, zero or positive, rather than a plain true/false — exists because the same function is reused later for **sorting** strings.

Knowing *which* of two strings comes first, not merely whether they are equal, is what makes alphabetical ordering possible. Here's the part worth flagging early, since it resurfaces later: a single function quietly does double duty for both equality-checking and ordering.

### 3.7.3 Hard-Coded Bounds and the Risk of a Segmentation Fault

Both `search0.c` and `search1.c` hard-code their loop bounds — `7` and `6`, respectively — rather than computing them from the array itself.

This is flagged as poor style for two reasons. First, it requires the programmer to count the array's elements by hand.

Second, it silently invites a specific class of bug if that count is ever miscopied.

A **segmentation fault** occurs when a program touches a part of memory it does not have permission to access — for instance, reading or writing past the last valid index of an array.

Consider what happens if `search1.c`'s bound were mistakenly written as `i < 7`, carried over by analogy from `search0.c`'s seven-element array of integers, instead of the correct `i < 6`.

Tracing this faulty version:

| Step | `i` | `strings[i]` | Status |
|---|---|---|---|
| 1–5 | 0–4 | valid elements | normal comparisons |
| 6 | 5 | `"top hat"` (last valid element) | normal comparison |
| 7 | 6 | *out of bounds* | `strcmp(strings[6], s)` reads memory the array was never allocated to use |

`strings[6]` does not correspond to any of the six Monopoly pieces.

It is one locker past the end of a six-locker row, in space the program never reserved.

The call `strcmp(strings[6], s)` then operates on whatever unpredictable data happens to occupy that memory.

The program most likely crashes with a segmentation fault rather than silently producing a wrong answer — though that is itself a small mercy. A wrong answer that looks plausible would be far harder to catch than an outright crash.

> A segmentation fault is the symptom, not the cause.
>
> The underlying cause is almost always an index that has drifted outside an array's valid range — `0` through `n-1`.
>
> Checking the loop bound against the array's actual size is the first thing worth verifying when a segmentation fault appears.

**Cheat sheet — array bounds:**

| Array | Valid indices | Common bug |
|---|---|---|
| `int numbers[7]` | `0` to `6` | loop bound `i < 7` (correct) |
| `string strings[6]` | `0` to `5` | loop bound mistakenly `i < 7` instead of `i < 6` → reads `strings[6]` → segfault |
| General: `T arr[n]` | `0` to `n-1` | any bound `i <= n` or larger reads one or more elements past the end |

## 3.8 Arrays, Parallel Data, and the Case for `struct`

### 3.8.1 A Phone Book With Parallel Arrays

`phonebook0.c` reconstructs the Week 0 phone-book lookup using two parallel arrays:
```c
#include <cs50.h>
#include <stdio.h>
#include <string.h>

int main(void)
{
    string names[] = {"Kelly", "David", "John"};
    string numbers[] = {"+1-617-495-1000", "+1-617-495-1000", "+1-949-468-2750"};

    string name = get_string("Name: ");
    for (int i = 0; i < 3; i++)
    {
        if (strcmp(names[i], name) == 0)
        {
            printf("Found %s\n", numbers[i]);
            return 0;
        }
    }
    printf("Not found\n");
    return 1;
}
```

Both arrays are declared as `string`, even though phone numbers look numeric.

Phone numbers often contain punctuation — `+`, `-` — that an integer type cannot hold.

A leading zero in a phone number, common in regional dialling prefixes outside the US, would also be silently dropped if the number were stored as an integer, since leading zeros carry no mathematical value.

Storing the number as a string of characters preserves it intact, punctuation and leading zeros included.

Tracing a search for `"David"`: at `i = 0`, `strcmp(names[0], name)` compares `"Kelly"` against `"David"` and returns nonzero.

At `i = 1`, `strcmp(names[1], name)` compares `"David"` against `"David"` and returns `0`.

The program then reads `numbers[1]` — not `names[1]` again — and prints `"+1-617-495-1000"`.

This last step is the crux of how parallel arrays work, and also their central risk. `numbers[1]` is read only because `i` still equals `1` from the matching iteration on `names`.

Nothing in the language ties `names[1]` and `numbers[1]` together as a unit. They happen to land at the same index purely because both arrays were typed out in the same order.

The search itself is a linear scan, since names are not stored in sorted order, which rules out binary search.

The two arrays can be pictured as two separate, unlabelled rows of lockers standing side by side: row one holds names, row two holds numbers.

The only thing connecting locker 1 in row one to locker 1 in row two is that a human walks both rows in lockstep, one index at a time. Nothing physically welds the two rows together.

> Parallel arrays place the burden of synchronisation entirely on the programmer.
>
> This is harmless for a handful of hard-coded entries.
>
> It becomes risky at scale, though — hundreds, 1mn entries or any real database — since nothing stops the two arrays from drifting out of sync if one is ever resized, reordered or edited without the other.

### 3.8.2 Defining a Custom Type with `typedef struct`

C provides no built-in type that bundles a name and a number together.

Only primitives — `int`, `float`, `char`, `bool`, `string` — exist natively.

The `typedef struct` keyword pair defines a new aggregate type that combines several fields into one:
```c
typedef struct
{
    string name;
    string number;
} person;
```

This declares a new type, `person`, where every `person` has a `string name` and a `string number`.

Placing this declaration above `main` makes the type available both to `main` and to any function defined later in the file.

A `struct` plays the same role in C that one row of an R `data.frame` plays — a single record where each column holds a differently-typed field belonging to that same row.

A Python `dict` serves the same purpose too, bundling related fields under one name, and is a reasonable second analogy where the R framing feels less natural.

An array alone cannot fill this role, because an array can only hold values of a single type. A `string` array cannot also carry a paired `int` or `bool` field for each of its elements without a second, separately-tracked array.

### 3.8.3 A Phone Book With Structs

`phonebook1.c` replaces the two parallel arrays with a single array of `person`:
```c
#include <cs50.h>
#include <stdio.h>
#include <string.h>

typedef struct
{
    string name;
    string number;
} person;

int main(void)
{
    person people[3];

    people[0].name = "Kelly";
    people[0].number = "+1-617-495-1000";
    people[1].name = "David";
    people[1].number = "+1-617-495-1000";
    people[2].name = "John";
    people[2].number = "+1-949-468-2750";

    string name = get_string("Name: ");
    for (int i = 0; i < 3; i++)
    {
        if (strcmp(people[i].name, name) == 0)
        {
            printf("Found %s\n", people[i].number);
            return 0;
        }
    }
    printf("Not found\n");
    return 1;
}
```

Struct fields are accessed with dot notation: `people[i].name`, `people[i].number`.

Tracing this program for a search of `"David"`: `people[0]` is a single `person` record holding both `"Kelly"` and `"+1-617-495-1000"` together.

At `i = 0`, `strcmp(people[0].name, name)` compares `"Kelly"` against `"David"` and returns nonzero.

At `i = 1`, `strcmp(people[1].name, name)` compares `"David"` against `"David"` and returns `0`.

The program then reads `people[1].number` — a field belonging to the very same record that just matched, not a second array indexed by coincidence — and prints `"+1-617-495-1000"`.

The search loop is otherwise unchanged: still three elements, each now holding two related values instead of one.

This bundling of multiple fields into a single unit is called **encapsulation**: each element of `people` is a complete, self-contained record.

The related fields are also laid out contiguously in memory within each struct instance, rather than relying on convention to keep two independent arrays aligned. `people[1].name` and `people[1].number` sit beside each other in memory because they belong to the same `person`, not because an index happened to line them up.

> Bridge: a C `struct` corresponds to one row of an R `data.frame`, where each column is a differently-typed field of the same record.
>
> It also corresponds to a Python `dict` (or a `dataclass`) bundling `{"name": ..., "number": ...}`, which is a workable secondary analogy.
>
> The array of `person` in `phonebook1.c` is the C analogue of an R `data.frame` with one row per person — or, in Python terms, a list of dicts.

**Cheat sheet — parallel arrays vs. `struct`:**

| | Parallel arrays (`phonebook0.c`) | `struct` (`phonebook1.c`) |
|---|---|---|
| Storage | two separate arrays | one array of records |
| Linking mechanism | shared index, by convention only | fields bundled inside one struct instance |
| Risk | arrays can silently drift out of sync | none — fields cannot become separated |
| Access | `names[i]`, `numbers[i]` | `people[i].name`, `people[i].number` |
| High-level analogue | two vectors walked in lockstep | a data.frame's rows / a list of dicts |

## 3.9 Sorting

### 3.9.1 The Sorting Problem

Sorting, as a black box, takes unsorted input and produces sorted output: an input such as `7 2 5 4 1 6 0 3` should become `0 1 2 3 4 5 6 7`.

```text
input (unsorted)  →  [ SORT ]  →  output (sorted)
7 2 5 4 1 6 0 3                  0 1 2 3 4 5 6 7
```

Binary search's efficiency advantage over linear search depends entirely on the data having already been sorted.

This motivates examining how sorting itself can be done, and at what cost — the same black-box treatment already applied to searching in 3.3.1.

### 3.9.2 Selection Sort

**Selection sort** repeatedly scans the unsorted portion of an array for its smallest remaining value and swaps that value into place at the front of that portion.

In English:
```text
Repeat until no unsorted elements remain:
    Search the unsorted part of the data to find the smallest value
    Swap the smallest found value with the first element of the unsorted part
```

In array notation:
```text
For i from 0 to n-1
    Find smallest number between numbers[i] and numbers[n-1]
    Swap smallest number with numbers[i]
```

Traced on `7 2 5 4 1 6 0 3`: the smallest value overall, `0`, is found and swapped into index 0, displacing the `7` that was there.

This produces `0 2 5 4 1 6 7 3`.

The next pass searches only the remaining unsorted region, indices 1 through 7, finds `1` and swaps it into index 1.

This pattern continues. Each pass fixes exactly one more element and never revisits an already-placed position.

The algorithm tracks only a single "smallest found so far" variable while scanning, rather than memorising every value and position it has seen.

This is a deliberate **trade-off between memory and time** that recurs throughout algorithm design: more bookkeeping can sometimes buy fewer comparisons, and selection sort chooses to keep bookkeeping minimal instead.

Because an array's size is fixed at the time of allocation — like a row of lockers that cannot grow a new one — there is no spare slot to insert a value into.

The smallest value found must instead be *swapped* with whatever currently occupies the destination index.

A complete run on the six-element array `5 2 1 3 6 4` — the same array used in 3.9.3.2's bubble-sort trace — shows the process through to completion, including the case where the smallest unsorted value is already sitting at the front of the unsorted part:

| Step | Unsorted part | Smallest found | First unsorted element | Action | Array after |
|---|---|---|---|---|---|
| 1 | 5,2,1,3,6,4 | 1 | 5 | swap 1 ↔ 5 | `1 2 5 3 6 4` |
| 2 | 2,5,3,6,4 | 2 | 2 | swap 2 ↔ 2 (no-op) | `1 2 5 3 6 4` |
| 3 | 5,3,6,4 | 3 | 5 | swap 3 ↔ 5 | `1 2 3 5 6 4` |
| 4 | 5,6,4 | 4 | 5 | swap 4 ↔ 5 | `1 2 3 4 6 5` |
| 5 | 6,5 | 5 | 6 | swap 5 ↔ 6 | `1 2 3 4 5 6` |
| 6 | 6 | 6 | — | only element left | `1 2 3 4 5 6` |

> When the smallest unsorted value already sits at the front of the unsorted part, as in step 2 above, the "swap" is a no-op: the value is swapped with itself.
>
> It remains in place, but the pass still confirms it sorted — the bookkeeping still treats that position as settled, even though nothing moved.

**Cost.** The first pass makes `n-1` comparisons, the second makes `n-2`, and so on down to a final pass of one comparison:
```text
(n-1) + (n-2) + (n-3) + ... + 1 = n(n-1)/2 = (n² - n)/2 = n²/2 - n/2
```

Keeping only the dominant term, selection sort is **O(n²)**.

The pseudocode contains no check for whether the data is already sorted. Every pass performs the same scan regardless of input.

The best case is therefore also **Ω(n²)**, and since the upper and lower bounds coincide, selection sort is **Θ(n²)**. It always does the same amount of work, with no favourable case — being handed an already-sorted six-element array buys no speed-up at all over the worst-ordered one.

### 3.9.3 Bubble Sort

**Bubble sort** compares each pair of adjacent elements and swaps them if out of order, causing the largest unsorted value to "bubble" toward the end of the list one swap at a time over the course of a single pass.

Selection sort builds the sorted result left to right, smallest to largest.

Bubble sort builds it right to left, largest to smallest: each pass settles one more of the largest remaining values into place at the tail of the array, rather than the smallest value into place at its head.

Initial pseudocode:
```text
Repeat n times
    For i from 0 to n-2
        If numbers[i] and numbers[i+1] out of order
            Swap them
```

The inner loop stops at `n-2`, not `n-1`.

The comparison checks index `i` against `i+1`. Were `i` permitted to reach `n-1`, the last valid index, `i+1` would then equal `n` — one past the end of the array.

> Off-by-one errors of exactly this kind — an index `i+1` overflowing past the last valid index — are a common source of bugs in array traversal.

Traced on `7 2 5 4 1 6 0 3`, one full pass moves the `7` rightward via successive adjacent swaps:

```text
7 2 5 4 1 6 0 3
2 7 5 4 1 6 0 3   (swap 7,2)
2 5 7 4 1 6 0 3   (swap 7,5)
2 5 4 7 1 6 0 3   (swap 7,4)
2 5 4 1 7 6 0 3   (swap 7,1)
2 5 4 1 6 7 0 3   (swap 7,6)
2 5 4 1 6 0 7 3   (swap 7,0)
2 5 4 1 6 0 3 7   (swap 7,3)
```

Seven adjacent swaps in this single pass, one for every element the `7` had to cross to reach the far end of an eight-element array.

Continuing past that first pass, each subsequent pass bubbles the next-largest remaining value into place, one position further left of the previous one settled:

| Pass | Array after pass |
|---|---|
| 1 | `2 5 4 1 6 0 3 7` |
| 2 | `2 4 1 5 0 3 6 7` |
| 3 | `2 1 4 0 3 5 6 7` |
| 4 | `1 2 0 3 4 5 6 7` |
| 5 | `1 0 2 3 4 5 6 7` |
| 6 | `0 1 2 3 4 5 6 7` |
| 7 | `0 1 2 3 4 5 6 7` — no swaps occur |

The array reaches its fully sorted state after the sixth pass.

The unoptimised pseudocode above, however, has no way of knowing that.

It runs the seventh pass anyway, scanning all eight elements without making a single swap, because nothing in the pseudocode checks whether a pass produced any swaps at all.

This wasted final pass is exactly the inefficiency the early-exit optimisation in 3.9.3.1 is designed to eliminate.

It suffices to repeat the outer loop only `n-1` times rather than `n`. Once `n-1` elements have bubbled into place, the one remaining element is necessarily already correct.

**Cost.** Reading the cost directly from the pseudocode: the outer loop runs `n-1` times, the inner loop runs `n-1` times per outer iteration and the comparison and swap are each constant-time regardless of list size.

Treating the two loops as nested — the same shape as the row/column traversal used for printing a grid of characters — the total is:
```text
(n-1) × (n-1) = n² - 2n + 1
```

Dropping lower-order terms, bubble sort is **O(n²)** — the same upper bound as selection sort.

> Classifying bubble sort as **O(n²)** is not a way of reporting an exact step count for one specific array.
>
> It is a label for how much worse the algorithm gets as the list grows.
>
> Doubling the length of the list roughly quadruples the work, because every element ends up compared against roughly every other element. An eight-element list and a sixteen-element list make this concrete: the larger one does not take twice as long, it takes closer to four times as long.
>
> Tripling the length makes the algorithm roughly nine times slower.
>
> This "growing faster than the input itself" pattern is what "squared" denotes, and it holds regardless of the specific values stored in the array.
>
> The O(n²) classification functions as a warning rather than a measurement. The algorithm performs acceptably on a small list, but degrades sharply as the list grows — the comparison point worth knowing before choosing an algorithm for a large dataset.

#### 3.9.3.1 Early-Exit Optimization

Bubble sort can be enhanced: if a full pass makes no swaps, the list is already sorted, and the algorithm can quit immediately rather than running the remaining passes:
```text
Repeat n-1 times
    For i from 0 to n-2
        If numbers[i] and numbers[i+1] out of order
            Swap them
    If no swaps
        Quit
```

With this optimisation, the best case requires only a single pass through an already-sorted list, giving bubble sort a lower bound of **Ω(n)**.

Because the lower bound, Ω(n), no longer matches the upper bound, O(n²), bubble sort with the early-exit check can no longer be described with **Θ**.

Its average and worst cases remain on the order of n²; the optimisation improves only the best case.

#### 3.9.3.2 Bubble Sort: Tracking Termination with a Swap Counter

The early-exit check from 3.9.3.1 — "if no swaps, quit" — can be implemented concretely with a **swap counter**: a variable incremented once for every swap made during a pass, and reset to `0` at the start of each new pass.
```text
Set swap counter to a non-zero value
Repeat until the swap counter is 0:
    Reset swap counter to 0
    Look at each adjacent pair
    If two adjacent elements are not in order, swap them and add 1 to the swap counter
```

The counter is deliberately initialised to a nonzero placeholder, such as `-1`, before the loop begins.

Were it started at `0` instead, the "repeat until 0" condition would already be satisfied, and the algorithm would never run a single pass.

Once inside the loop, the counter is reset to `0` at the top of every pass and tracks only that pass's swaps.

If a full pass completes with the counter still at `0`, no adjacent pair was out of order, and the array is provably sorted.

Traced on a six-element array `5 2 1 3 6 4`:

| Pass | Swaps made | Array after pass | Swap counter |
|---|---|---|---|
| 1 | (5,2), (5,1), (5,3), (6,4) | `2 1 3 5 4 6` | 4 |
| 2 | (2,1), (5,4) | `1 2 3 4 5 6` | 2 |
| 3 | none | `1 2 3 4 5 6` | 0 → stop |

Four swaps on the first pass, two on the second, zero on the third — the counter itself traces the algorithm's convergence toward a sorted array.

The third pass, which makes no swaps, is what proves the array sorted and terminates the loop.

This is the same mechanism as 3.9.3.1's "if no swaps, quit," made concrete here with a counted variable rather than a boolean flag.

> The worst case — a fully reverse-sorted array — can also be understood spatially.
>
> Every element may need to move across the *entire* array: large elements bubble all the way to the right, small elements bubble all the way to the left.
>
> Each of the `n` elements can therefore require up to `n` steps of movement.
>
> This reinforces the O(n²) bound alongside the `(n-1) × (n-1)` algebraic derivation already given — two different ways of arriving at the same conclusion, one spatial and one algebraic.

**Cheat sheet — selection sort vs. bubble sort:**

| | Selection sort | Bubble sort |
|---|---|---|
| Builds result | left to right (smallest first) | right to left (largest first) |
| Per-pass action | find global minimum of unsorted part, swap once | compare every adjacent pair, swap each time out of order |
| Best case | Ω(n²) — always rescans fully | Ω(n) — with early exit, one clean pass |
| Worst case | O(n²) | O(n²) |
| Has Θ? | yes, Θ(n²) | no — once early exit is added, best and worst cases diverge |
| Termination check | none needed; loop bound is fixed | swap counter, or boolean flag, enables early exit |

## 3.10 Recursion

### 3.10.1 Recursive Functions, Base Cases, and Recursive Cases

**Recursion** solves a problem by having a function call itself.

A **recursive function** is one defined in terms of itself, mirroring a mathematical definition of the form `f(x) = f(something)`.

> A function calling itself appears to risk an infinite loop.
>
> Nothing obviously stops the calls from continuing indefinitely.

Two terms resolve this concern.

A **base case** is a condition answerable immediately, with no further recursive work required. It is what stops the recursion.

A **recursive case** is the part of the function that calls itself again with a modified, strictly smaller input.

Each recursive call moves the problem closer to the base case. So long as every recursive call shrinks the problem, the recursion eventually reaches the base case and terminates.

A call that handed off a same-sized problem would never make progress, and would never terminate.

Each of these two units — the base case and the recursive case — is established here in isolation, before being combined into worked examples.

A base case answers the question directly. A recursive case restates the same question on a smaller input and trusts the function to answer that smaller question correctly. Here's the part that's easy to miss: "trusting" a smaller call to produce the right answer is not blind faith — it follows from the same definition applying at every size, base case included.

#### 3.10.1.1 The Factorial Function: A Worked Recursive/Iterative Comparison

The mathematical **factorial** function, conventionally written `n!`, multiplies together every positive integer from `n` down to `1`.

For example, `5! = 5 × 4 × 3 × 2 × 1`.

Written as a function `fact(n)`, the pattern is suggestive: `fact(2) = 2 × fact(1)`, `fact(3) = 3 × fact(2)`, `fact(4) = 4 × fact(3)`, and so on. Each value is `n` times the factorial of `n - 1`.

Generalising:

```text
fact(n) = n * fact(n-1)
```

This formula supplies a base case and a recursive case.

The base case is `fact(1) = 1`, which needs no further multiplication. The recursive case covers every other `n`, expressed in terms of a smaller call to the same function:

```c
int fact(int n)
{
    if (n == 1)
        return 1;
    else
        return n * fact(n-1);
}
```

The equivalent iterative version uses a loop and an accumulator variable instead of a self-call:

```c
int fact2(int n)
{
    int product = 1;
    while (n > 0)
    {
        product *= n;
        n--;
    }
    return product;
}
```

Both versions compute the same result.

The recursive version expresses the *definition* of factorial almost directly in code. The iterative version instead explicitly manages a running product and a countdown.

The same trade-off reappears later between `recursion.c` and `iteration.c`'s pyramid-drawing functions.

R is the more useful bridge language here, since its recursive-function syntax maps onto C's almost line for line. An R translation:

```r
fact <- function(n) {
  if (n == 1) return(1)
  else return(n * fact(n - 1))
}
```

The shape is unmistakably the same as the C version: one `if` for the base case, one `else` for the recursive case and a self-call passing a strictly smaller argument. R's `function` keyword replaces C's type-and-name declaration, and `return()` is written as a call rather than a bare keyword, but nothing about the recursive logic itself changes.

A direct Python translation makes the same logic concrete in a second high-level language already familiar:

```python
def fact(n):
    if n == 1:
        return 1
    else:
        return n * fact(n - 1)
```

The structure is identical to both the C and R versions: one `if` for the base case, one `else` for the recursive case and a self-call passing a strictly smaller argument.

Python requires no type declarations and no explicit return type, but the recursive shape of the logic does not change. The same is true of R — neither high-level language forces the writer to think about memory the way C eventually will, but the recursive *idea* transfers across all three languages untouched.

#### 3.10.1.2 Multiple Base Cases: The Fibonacci Sequence

A recursive function is not limited to a single base case or a single recursive case.

The **Fibonacci sequence** — 0, 1, 1, 2, 3, 5, 8, ... — illustrates a function with two base cases.

The first element is defined directly as `0`. The second element is defined directly as `1`. Every later element is the sum of the two preceding it.

```text
If n is 1, return 0.
If n is 2, return 1.
Otherwise, return fibonacci(n-1) + fibonacci(n-2).
```

Here, two separate conditions — `n == 1` and `n == 2` — each stop the recursion immediately, rather than just one.

Both are base cases; only the third branch is recursive. It is worth restating plainly why two base cases are needed rather than one: a single base case at `n == 1` alone would have nothing to anchor `n == 2`'s "sum of the two before it," since there would be no defined element at position 0 to add.

#### 3.10.1.3 Multiple Recursive Cases: The Collatz Conjecture

A function can likewise have more than one recursive case.

The **Collatz conjecture** speculates that, for any positive integer `n`, repeatedly applying a simple rule always eventually reaches `1`:

```text
If n is 1, stop.
Otherwise, if n is even, repeat this process on n / 2.
Otherwise, if n is odd, repeat this process on 3n + 1.
```

Counting how many steps this takes to reach `1` gives a single base case (`n == 1`, `0` steps) alongside *two* distinct recursive cases: one for even `n`, one for odd `n`.

```c
int collatz(int n)
{
    // base case
    if (n == 1)
        return 0;
    // even numbers
    else if ((n % 2) == 0)
        return 1 + collatz(n/2);
    // odd numbers
    else
        return 1 + collatz(3*n + 1);
}
```

Worked examples: `collatz(2)` takes 1 step (`2 → 1`).

`collatz(3)` takes 7 steps (`3 → 10 → 5 → 16 → 8 → 4 → 2 → 1`).

`collatz(7)` takes 16 steps (`7 → 22 → 11 → 34 → 17 → 52 → 26 → 13 → 40 → 20 → 10 → 5 → 16 → 8 → 4 → 2 → 1`). Each of these three traces is the same single function running on a different starting number — the branching logic never changes, only how many times the even or odd rule fires before `n` happens to land on `1`.

> No proof exists that the process always terminates for every positive integer.
>
> The conjecture has nonetheless been verified across enormous ranges of numbers.

This illustrates that a recursive function's *correctness* — whether it computes the right answer — and its *termination* — whether it provably always reaches a base case — are separate questions.

The Collatz recursion above terminates for every input ever checked, without there existing a general proof that it always must.

An R translation preserves the same three-way branch:

```r
collatz <- function(n) {
  if (n == 1) return(0)
  else if (n %% 2 == 0) return(1 + collatz(n / 2))
  else return(1 + collatz(3 * n + 1))
}
```

R's modulo operator is `%%`, not C's `%`, and integer-valued division here works without a separate operator the way Python needs `//` — dividing two whole numbers in R that happen to come out even still returns a usable numeric value. The branching structure underneath is otherwise the same three cases as the C version.

A direct Python translation also preserves the same branching structure:

```python
def collatz(n):
    if n == 1:
        return 0
    elif n % 2 == 0:
        return 1 + collatz(n // 2)
    else:
        return 1 + collatz(3 * n + 1)
```

Integer division uses `//` in Python rather than C's `/` on `int` operands, but the recursive branching is otherwise unchanged across all three languages.

### 3.10.2 Recursion Revisited: The Phone-Book Search

The Week 0 phone-book search, and the locker search earlier in this lecture, are reframed here as the course's first (implicit) examples of recursion.

The original procedural formulation used explicit "go back to" jumps to induce looping:

```text
1  Pick up phone book
2  Open to middle of phone book
3  Look at page
4  If person is on page
5      Call person
6  Else if person is earlier in book
7      Open to middle of left half of book
8      Go back to line 3
9  Else if person is later in book
10     Open to middle of right half of book
11     Go back to line 3
12 Else
13     Quit
```

This collapses into a recursive formulation by treating lines 7–8 — "open to the middle of the left half," then "go back to line 3" — as a single instruction: "search the left half of the book," restating the whole algorithm on a smaller book.

Lines 10–11 collapse the same way into "search the right half of the book":

```text
1  Pick up phone book
2  Open to middle of phone book
3  Look at page
4  If person is on page
5      Call person
6  Else if person is earlier in book
7      Search left half of book
9  Else if person is later in book
10     Search right half of book
12 Else
13     Quit
```

> Line numbers 8 and 11 are intentionally absent here, not renumbered away. They mark exactly which steps each recursive call subsumes.
>
> "Search left half of book" *is* "open to the middle of the left half, look at the page, and (if necessary) go back to line 3," collapsed into one self-referential instruction, since the procedure being invoked is the very same search.

Lines 7 and 10 are now recursive calls: the same algorithm invoked again on a halved problem, with no explicit jump to a named line.

The base case is the check at line 4 (found) or line 12/13 (none left, quit). The recursive case is searching the left or right half, each strictly smaller than the last.

### 3.10.3 A Recursive Definition: the Pyramid

A half-pyramid (the shape built in Problem Set 1) can itself be defined recursively.

A pyramid of height 4 is a pyramid of height 3 plus one more row. Height 3 is height 2 plus one more row. Height 2 is height 1 plus one more row.

Height 1 is a single brick, stated directly rather than in terms of a smaller pyramid — this is the **base case**.

Every other height is the **recursive case**, defined in terms of the same structure, one row shorter.

### 3.10.4 Drawing a Pyramid Iteratively

`iteration.c` draws a pyramid using nested loops:

```c
#include <cs50.h>
#include <stdio.h>

void draw(int n);

int main(void)
{
    int height = get_int("Height: ");
    draw(height);
}

void draw(int n)
{
    for (int i = 0; i < n; i++)
    {
        for (int j = 0; j < i + 1; j++)
        {
            printf("#");
        }
        printf("\n");
    }
}
```

The inner bound is `i + 1`, not `i`.

Row 0 should print one brick, and since `i` starts at 0, the inner loop must run from `j = 0` to `j < 1` to produce exactly one `#` on that row. Successive rows then yield 1, 2, 3, 4 bricks.

> A first attempt at compiling failed because `draw` was called in `main` before being fully defined later in the file, with no function prototype declared at the top.
>
> Adding `void draw(int n);` above `main` resolves it: the compiler then knows `draw`'s signature before it is used, even though the full definition appears later in the file.

**Iteration** is defined plainly here as using loops to solve a problem.

### 3.10.5 Drawing a Pyramid Recursively

```c
void draw(int n)
{
    if (n <= 0)
        return;
    draw(n - 1);
    for (int i = 0; i < n; i++)
        printf("#");
    printf("\n");
}
```

In C, a single-line `if` body permits the curly braces to be omitted. That is why `if (n <= 0) return;` is written without braces here.

**The base case.** The check `if (n <= 0)` exists to stop the recursion.

Without it, the function would call itself forever; modern compilers (such as `clang`) refuse to compile a function for which every path leads to a call to itself, flagging it as infinite recursion.

The comparison uses `<= 0` rather than `== 0` as a defensive measure: if the function were called with a negative argument (e.g. `draw(-5)`), it still terminates immediately rather than running past zero indefinitely.

> The base case is the safety brake. The recursive case is the engine that drives the function toward that brake.
>
> Neither is sufficient alone — a function with only a recursive case never stops, and a function with only a base case never does anything beyond the trivial input.

**The call stack.** When a function calls itself, the computer does not jump around arbitrarily; it tracks every paused call using a **call stack**.

A call stack behaves like a stack of plates, or a pile of sticky notes left on a desk: each call pushes one note on top of the pile, recording where execution should resume once that call finishes.

Each return pops the topmost note off the pile, exposing the one directly beneath it — which is necessarily the call that pushed it in the first place.

Applied to `draw`: when `draw(3)` reaches its recursive call, it calls `draw(2)`.

Before control transfers, a note is effectively written: "paused in `draw(3)`, resume at the `for` loop upon return." That note goes on top of the stack.

The same happens as `draw(2)` calls `draw(1)`, and as `draw(1)` calls `draw(0)` — each call adds one more note to the pile, and the computer only ever acts on the topmost note.

When `n` reaches `0`, the base case fires and the function returns immediately, popping its own note off the stack and exposing the note directly beneath it: `draw(1)`, paused and waiting to resume exactly where it left off.

> Each frame's local variables are isolated from every other frame.
>
> The loop variable `i` inside `draw(3)` is a separate piece of memory from the `i` inside `draw(2)` or `draw(1)`; no frame can see or overwrite another's variables.
>
> This isolation is part of what the call stack tracks alongside each resume point.

#### 3.10.5.1 Tracing `draw(3)`: Calls Descend Before Any Printing Begins

Tracing `draw(3)` line by line makes explicit when each call happens, when the base case fires, and — critically — that printing happens only as calls return, not as they descend.

`draw(3)` is called with `n = 3`. It checks `n <= 0`, which is false, and proceeds to call `draw(2)`.

This pauses `draw(3)` at that line until `draw(2)` returns.

`draw(2)` is called with `n = 2`. It checks `n <= 0`, which is false, and calls `draw(1)`, pausing until `draw(1)` returns.

`draw(1)` is called with `n = 1`. It checks `n <= 0`, which is false, and calls `draw(0)`, pausing until `draw(0)` returns.

`draw(0)` is called with `n = 0`. It checks `n <= 0`, which is true this time.

It returns immediately, without printing anything. This is the deepest point the recursion reaches — four calls stacked up, none of them having printed a single character yet.

Control now returns to `draw(1)`, which resumes right after its own `draw(n - 1)` line.

Its `for` loop runs once (`i = 0`), printing one `#`. The subsequent `printf("\n")` ends that row.

`draw(1)` finishes and returns to `draw(2)`.

`draw(2)` resumes after its `draw(n - 1)` line. Its `for` loop runs twice (`i = 0, 1`), printing two `#` characters, followed by a newline that ends that row.

`draw(2)` finishes and returns to `draw(3)`.

`draw(3)` resumes after its `draw(n - 1)` line. Its `for` loop runs three times (`i = 0, 1, 2`), printing three `#` characters, followed by a newline.

`draw(3)` finishes.

The resulting output is produced in exactly this order:

```text
#
##
###
```

> The calls descend in the order 3 → 2 → 1 → 0, reaching the base case before anything prints.
>
> Printing then happens in the reverse order, 1 → 2 → 3, on the way back out.
>
> This reversal is why the shortest row prints first and the longest row prints last, building the pyramid top-to-bottom even though the function called itself "smallest last."

The call-stack analogy explains why the reversal occurs.

Each call pushes a new note (or plate) recording its own value of `n` and exactly where execution should resume.

Returning always pops the most recently pushed note, exposing whatever sits directly beneath it — and that note is necessarily the call that pushed it.

`draw(0)`'s note sits on top, so it is popped first, exposing `draw(1)`'s note; `draw(1)`'s note is popped next, exposing `draw(2)`'s; and so on.

Each frame's print statement only runs once its note is the one being resumed, which happens in the opposite order from how the notes were laid down. Here's the part that's easy to miss on a first read: nothing about the *code* explicitly reverses anything — the reversal falls out entirely from how a stack pops in last-in-first-out order.

An R translation preserves the same recursive shape, including a driver that reads a height and calls the function:

```r
draw <- function(n) {
  if (n <= 0) return(invisible(NULL))
  draw(n - 1)
  cat(strrep("#", n), "\n", sep = "")
}

main <- function() {
  height <- as.integer(readline("Height: "))
  draw(height)
}

main()
```

R's `cat()`, like C's `printf`, does not append a newline automatically, so the row of `#` characters and the line break that ends it must both be written out explicitly — here folded into a single `cat()` call via `strrep("#", n)` followed by `"\n"`, with `sep = ""` so the two pieces are not separated by `cat()`'s default space.

A direct Python translation also preserves the same structure, including a driver that reads a height and calls the function:

```python
def draw(n):
    if n <= 0:
        return
    draw(n - 1)
    print("#" * n)


def main():
    height = int(input("Height: "))
    draw(height)


main()
```

`print()` in Python appends a newline automatically, so a single call to `print("#" * n)` produces one full row followed by a line break — unlike the C version's bare `printf("#")` inside a loop, which requires a separate `printf("\n")` afterward to end the row, and unlike R's `cat()`, which needs the newline supplied by hand as well.

## 3.11 Merge Sort

### 3.11.1 Pseudocode and the Merge Step

**Merge sort**, a third sorting algorithm, uses recursion to make far fewer comparisons than selection or bubble sort.

It avoids repeated work on the same elements.

Pseudocode:

```text
Sort the left half of the numbers
Sort the right half of the numbers
Merge the sorted halves
```

This is self-referential: the first two lines invoke the same procedure on smaller inputs, making it recursive by construction.

The base case is a list of size 0 or 1, which is already sorted and requires no further work.

**Merging** two already-sorted halves of equal size proceeds by comparing the foremost unconsumed element of each half, taking the smaller of the two and advancing only the pointer belonging to the half that contributed it.

Given a left half `1, 3, 4, 6` and a right half `0, 2, 5, 7`:

| Compare | Take | Advance |
|---|---|---|
| 1 vs 0 | 0 | right |
| 1 vs 2 | 1 | left |
| 3 vs 2 | 2 | right |
| 3 vs 5 | 3 | left |
| 4 vs 5 | 4 | left |
| 6 vs 5 | 5 | right |
| 6 vs 7 | 6 | left (left exhausted) |
| — | 7 | remaining element |

The result is `0, 1, 2, 3, 4, 5, 6, 7`.

Unlike selection or bubble sort, the merge pointers move only forward, touching each element exactly once. Merging two sorted halves totaling `n` elements takes `n` steps.

### 3.11.2 Tracing Merge Sort on an 8-Element List

Starting with the unsorted list `[6, 3, 4, 1, 5, 2, 0, 7]`, merge sort uses recursion to divide the list before sorting and merging it back together.

The splitting and merging mirrors sorting a messy stack of 8 test scores.

The stack is split in half repeatedly until 8 individual stacks of one paper each remain; a single paper is sorted by definition, so this is the base case. The stacks are then merged back together two at a time, sorted as they combine.

**Phase 1: descending through the splits.** The list is divided in half recursively.

The left side is split all the way down to its base cases before the right side is processed. This left-before-right ordering matters for tracing the recursion correctly, even though the final sorted output would come out the same either way:

1. Start: `[6, 3, 4, 1, 5, 2, 0, 7]`
2. Split 1 (left half): `[6, 3, 4, 1]`
   - Split 2 (left-left): `[6, 3]`
     - Split 3 (base cases): `[6]` and `[3]` — neither can split further; merging begins.
   - Split 4 (left-right): `[4, 1]`
     - Split 5 (base cases): `[4]` and `[1]` — merging begins.
3. Split 6 (right half): `[5, 2, 0, 7]`
   - Split 7 (right-left): `[5, 2]`
     - Split 8 (base cases): `[5]` and `[2]`.
   - Split 9 (right-right): `[0, 7]`
     - Split 10 (base cases): `[0]` and `[7]`.

**Phase 2: ascending through the merges.** Once the base cases are reached, the recursion unwinds, merging the sub-lists back together in sorted order.

Merge level 1 combines individual elements into pairs:
- `[6]` and `[3]` merge into `[3, 6]`
- `[4]` and `[1]` merge into `[1, 4]`
- `[5]` and `[2]` merge into `[2, 5]`
- `[0]` and `[7]` merge into `[0, 7]`

Merge level 2 combines 2-element lists into 4-element lists:
- `[3, 6]` and `[1, 4]` merge into `[1, 3, 4, 6]`
- `[2, 5]` and `[0, 7]` merge into `[0, 2, 5, 7]`

Merge level 3 combines the two 4-element lists into the final 8-element list:
- `[1, 3, 4, 6]` and `[0, 2, 5, 7]` merge into `[0, 1, 2, 3, 4, 5, 6, 7]`

Each merge level only ever combines lists that are already fully sorted on their own, which is exactly why a simple left-to-right comparison at each step is enough — there is never a need to look back at an element already taken.

The full execution can be visualised as a tree, splitting down to single elements and merging back up to the sorted whole:

```text
              [6, 3, 4, 1, 5, 2, 0, 7]             <- Unsorted Start
                    /          \
            [6, 3, 4, 1]      [5, 2, 0, 7]         <- Split 1
             /        \        /        \
          [6, 3]    [4, 1]   [5, 2]    [0, 7]      <- Split 2
          /    \    /    \   /    \    /    \
        [6]   [3]  [4]   [1][5]   [2] [0]   [7]    <- Base Cases (Size 1)
        =======================================
          \    /    \    /   \    /    \    /
          [3, 6]    [1, 4]   [2, 5]    [0, 7]      <- Merge Level 1
             \        /        \        /
            [1, 3, 4, 6]      [0, 2, 5, 7]         <- Merge Level 2
                    \          /
              [0, 1, 2, 3, 4, 5, 6, 7]             <- Final Sorted List
```

#### 3.11.2.1 Tracing Merge Sort on a Six-Element List (Odd Split)

The 8-element trace above always splits evenly, since 8, 4 and 2 are all even.

A six-element array, `5, 2, 1, 3, 6, 4` — the same array used in 3.9.2's selection-sort trace and 3.9.3.2's bubble-sort trace — produces a sub-array of odd length partway through, illustrating how merge sort handles a split that cannot be even.

When a sub-array has an odd number of elements, the convention is to make the **left half the smaller of the two** (e.g., splitting 3 elements into a 1-element left half and a 2-element right half).

The specific choice is arbitrary. What matters is applying it **consistently** throughout the recursion, since merge sort's correctness does not depend on how a tie in size is broken, only on always breaking it the same way.

Tracing `5, 2, 1, 3, 6, 4`:

1. **Sort the left half** (`5, 2, 1`). This splits unevenly into left `5` (a base case) and right `2, 1` (split further into `2` and `1`, merged into `1, 2`). Merging `5` and `1, 2` proceeds as follows: compare `5` vs `1`, take `1`; compare `5` vs `2`, take `2`; nothing remains on the right, so `5` is taken last. The result is `1, 2, 5`.
2. **Sort the right half** (`3, 6, 4`). This splits unevenly into left `3` (a base case) and right `6, 4` (split into `6` and `4`, merged into `4, 6`). Merging `3` and `4, 6` proceeds as follows: compare `3` vs `4`, take `3`; nothing remains on the left, so the entire remaining right side (`4, 6`) is taken as-is. The result is `3, 4, 6`.
3. **Merge the two sorted halves**, `1, 2, 5` and `3, 4, 6`, comparing element by element: `1` vs `3` takes `1`; `2` vs `3` takes `2`; `5` vs `3` takes `3`; `5` vs `4` takes `4`; `5` vs `6` takes `5`; `6` vs nothing takes `6`. This produces the final sorted list `1, 2, 3, 4, 5, 6`.

> When one side of a merge is exhausted, the comparison becomes "the remaining value vs. nothing."
>
> The remaining side's values are necessarily smaller (or larger) than everything already merged, since both sides arrived at the merge step already sorted.
>
> They are taken in order without further comparison.

### 3.11.3 Big-O Analysis: Θ(n log n)

For `n = 8`, the algorithm operates at 3 distinct levels, and the total merging work across all sublists at any one level is `n` steps.

The number of levels relates to `n` logarithmically: `8 = 2³`, so `log₂8 = 3`.

In general, an input of size `n` splits into `log₂n` levels, written `log n` with the base omitted under Big-O. With `n` steps of work at each of `log n` levels, the total running time is on the order of **n log n**.

| Bound | Value |
|---|---|
| Big O (upper bound) | O(n log n) |
| Big Ω (lower bound) | Ω(n log n) |
| Big Θ | Θ(n log n) |

> Merge sort has no best-case shortcut analogous to bubble sort's early exit, so its lower bound matches its upper bound in all cases.
>
> Even an already-sorted array must still be fully split and recombined, since the algorithm has no way to detect sortedness in advance.
>
> This is a strictly better asymptotic running time than either selection sort or bubble sort (both O(n²)), at the cost of roughly twice the memory: an extra buffer is needed to hold elements during each merge.
>
> This is a classic **time/space trade-off**.

### 3.11.4 Comparing All Three Sorting Algorithms

A side-by-side visualisation sorting the same randomised data with all three algorithms running at the same speed shows merge sort finishing visibly faster than bubble sort or selection sort.

Dividing the problem recursively touches far fewer total elements than repeatedly rescanning the whole list.

| Algorithm | Big O | Big Ω | Big Θ | Improvement #1 |
|---|---|---|---|---|
| Selection sort | O(n²) | Ω(n²) | Θ(n²) | — |
| Bubble sort | O(n²) | Ω(n) | — | Early exit when a pass makes no swaps |
| Merge sort | O(n log n) | Ω(n log n) | Θ(n log n) | — |

> **Cheat sheet — recursion vocabulary**
>
> | Term | Meaning |
> |---|---|
> | Recursive function | A function defined in terms of a call to itself |
> | Base case | The condition that stops recursion; answered directly, with no further self-call |
> | Recursive case | The branch that calls the function again on a strictly smaller input |
> | Call stack | The structure tracking every paused call, most recent on top, in resume order |
> | Stack frame | One call's isolated record on the call stack: its arguments, local variables and resume point |
> | Stack overflow | The crash that results when recursion never reaches its base case and the call stack exceeds available memory |
>
> **Cheat sheet — sorting algorithms compared**
>
> | Algorithm | Strategy | Big O | Big Ω | Big Θ |
> |---|---|---|---|---|
> | Selection sort | Repeatedly select the smallest remaining element | O(n²) | Ω(n²) | Θ(n²) |
> | Bubble sort | Repeatedly swap adjacent out-of-order pairs | O(n²) | Ω(n) | — (no single Θ; best case differs from worst) |
> | Merge sort | Recursively split, then merge sorted halves | O(n log n) | Ω(n log n) | Θ(n log n) |